In [1]:
import urllib.request
import os
import pandas as pd
from datetime import datetime

Для кожної з адміністративних одиниць України завантажити (urllib) тестові структуровані файли, що містять значення VHI-індексу. При зберіганні файлу, до його імені потрібно додати дату та час завантаження. Передбачити повторні запуски скрипту, реалізувати механізм запобігання повторного довантаження та колізії даних.

In [2]:
# Завантаження VHI-файлів для всіх 27 областей України
# Запобігання повторному завантаженню: перевіряємо наявність файлу з тим самим provinceID

DATA_DIR = "vhi_data"
os.makedirs(DATA_DIR, exist_ok=True)

def download_vhi_files():
    for province_id in range(1, 28):
        # Перевірка: чи вже є файл для цієї області
        existing = [f for f in os.listdir(DATA_DIR) if f.startswith(f"vhi_{province_id}_")]
        if existing:
            print(f"Область {province_id}: вже завантажено ({existing[0]}), пропускаємо.")
            continue

        url = (
            f"https://www.star.nesdis.noaa.gov/smcd/emb/vci/VH/get_TS_admin.php"
            f"?country=UKR&provinceID={province_id}&year1=1981&year2=2024&type=Mean"
        )
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        filename = os.path.join(DATA_DIR, f"vhi_{province_id}_{timestamp}.csv")

        try:
            urllib.request.urlretrieve(url, filename)
            print(f"Область {province_id}: завантажено → {filename}")
        except Exception as e:
            print(f"Область {province_id}: помилка — {e}")

download_vhi_files()

Область 1: вже завантажено (vhi_1_20260314_193741.csv), пропускаємо.
Область 2: вже завантажено (vhi_2_20260314_193746.csv), пропускаємо.
Область 3: вже завантажено (vhi_3_20260314_193749.csv), пропускаємо.
Область 4: вже завантажено (vhi_4_20260314_193751.csv), пропускаємо.
Область 5: вже завантажено (vhi_5_20260314_193753.csv), пропускаємо.
Область 6: вже завантажено (vhi_6_20260314_193756.csv), пропускаємо.
Область 7: вже завантажено (vhi_7_20260314_193758.csv), пропускаємо.
Область 8: вже завантажено (vhi_8_20260314_193800.csv), пропускаємо.
Область 9: вже завантажено (vhi_9_20260314_193802.csv), пропускаємо.
Область 10: вже завантажено (vhi_10_20260314_193803.csv), пропускаємо.
Область 11: вже завантажено (vhi_11_20260314_193805.csv), пропускаємо.
Область 12: вже завантажено (vhi_12_20260314_193806.csv), пропускаємо.
Область 13: вже завантажено (vhi_13_20260314_193807.csv), пропускаємо.
Область 14: вже завантажено (vhi_14_20260314_193809.csv), пропускаємо.
Область 15: вже завантаж

Зчитати завантажені текстові файли у pandas dataframe. Здійснити data cleaning: прибрати зайві стовпці, заповнити пропуски, видалити зайвий текст тощо. Додати стовпчики з назвою та індексом області

In [3]:
NOAA_TO_UA = {
    1:  (1,  "Черкаська"),
    2:  (2,  "Чернігівська"),
    3:  (3,  "Чернівецька"),
    4:  (4,  "Кримська"),
    5:  (5,  "Дніпропетровська"),
    6:  (6,  "Донецька"),
    7:  (7,  "Івано-Франківська"),
    8:  (8,  "Харківська"),
    9:  (9,  "Херсонська"),
    10: (10, "Хмельницька"),
    11: (11, "Київська"),
    12: (12, "Київ (місто)"),
    13: (13, "Кіровоградська"),
    14: (14, "Луганська"),
    15: (15, "Львівська"),
    16: (16, "Миколаївська"),
    17: (17, "Одеська"),
    18: (18, "Полтавська"),
    19: (19, "Рівненська"),
    20: (20, "Севастополь"),
    21: (21, "Сумська"),
    22: (22, "Тернопільська"),
    23: (23, "Закарпатська"),
    24: (24, "Вінницька"),
    25: (25, "Волинська"),
    26: (26, "Запорізька"),
    27: (27, "Житомирська"),
}

def load_all_vhi(data_dir=DATA_DIR):
    frames = []
    for fname in sorted(os.listdir(data_dir)):
        if not fname.endswith(".csv"):
            continue

        # Визначаю noaa_id з імені файлу 
        noaa_id = int(fname.split("_")[1])
        fpath = os.path.join(data_dir, fname)

        # Читаю всі рядки файлу
        with open(fpath, "r") as f:
            lines = f.readlines()

        # Шукаю рядок-заголовок — без урахування регістру, бо в файлах NOAA заголовок написаний маленькими літерами
        header_idx = next(
            (i for i, l in enumerate(lines) if "year" in l.lower()), None
        )

        if header_idx is None:
            print(f"Не знайшов заголовок у {fname}, пропускаю")
            continue

        # Прибираю HTML-теги з рядка заголовку
        lines[header_idx] = (
            lines[header_idx]
            .replace("<pre>", "")
            .replace("<br>", "")
            .strip() + "\n"
        )

        # Збираю валідні рядки з даними — прибираю <tt><pre> з першого рядка даних та <br> з решти, пропускаю закриваючі HTML-теги
        clean_lines = [lines[header_idx]] + [
            l.replace("<tt><pre>", "").replace("<br>", "")
            for l in lines[header_idx + 1:]
            if not l.strip().startswith("</") and l.strip() != ""
        ]

        # Парсю як CSV з index_col=False — бо в кінці кожного рядка є зайва кома
        from io import StringIO
        df = pd.read_csv(StringIO("".join(clean_lines)), index_col=False)

        # Прибираю пробіли з назв колонок
        df.columns = df.columns.str.strip()

        # Приводжу назви колонок до єдиного регістру
        df.columns = [c.capitalize() for c in df.columns]

        if "Year" not in df.columns:
            print(f"Колонки у {fname}: {df.columns.tolist()}, пропускаю")
            continue

        # Залишаю тільки потрібні колонки
        needed = ["Year", "Week", "Smn", "Smt", "Vci", "Tci", "Vhi"]
        df = df[[c for c in needed if c in df.columns]]

        # Перейменовую колонки у верхній регістр для зручності
        df.columns = [c.upper() for c in df.columns]

        # Видаляю рядки з пропущеними значеннями VHI
        df = df.dropna(subset=["VHI"])

        # Приводжу типи даних
        df["YEAR"] = pd.to_numeric(df["YEAR"], errors="coerce")
        df["WEEK"] = pd.to_numeric(df["WEEK"], errors="coerce")
        df["VHI"]  = pd.to_numeric(df["VHI"],  errors="coerce")

        # Прибираю рядки де не вдалось розпарсити числа
        df = df.dropna(subset=["YEAR", "WEEK", "VHI"])

        # Прибираю рядки з sentinel-значенням -1 
        df = df[df["VHI"] != -1]

        # Додаю інформацію про область
        ua_id, ua_name = NOAA_TO_UA.get(noaa_id, (noaa_id, "Невідома"))
        df["noaa_id"] = noaa_id
        df["ua_id"]   = ua_id
        df["region"]  = ua_name

        frames.append(df)

    if not frames:
        print("Не завантажив жодного файлу")
        return pd.DataFrame()

    # Об'єдную всі області в один датафрейм
    return pd.concat(frames, ignore_index=True)

df = load_all_vhi()
print(df.shape)
df.head(10)

(59022, 10)


,YEAR,WEEK,SMN,SMT,VCI,TCI,VHI,noaa_id,ua_id,region
0,1982,1,0.059,258.24,51.11,48.78,49.95,10,10,Хмельницька
1,1982,2,0.063,261.53,55.89,38.20,47.04,10,10,Хмельницька
2,1982,3,0.063,263.45,57.30,32.69,44.99,10,10,Хмельницька
3,1982,4,0.061,265.10,53.96,28.62,41.29,10,10,Хмельницька
4,1982,5,0.058,266.42,46.87,28.57,37.72,10,10,Хмельницька
5,1982,6,0.056,267.47,39.55,30.27,34.91,10,10,Хмельницька
6,1982,7,0.055,268.58,35.19,31.10,33.14,10,10,Хмельницька
7,1982,8,0.057,270.15,33.35,32.09,32.72,10,10,Хмельницька
8,1982,9,0.057,271.60,30.82,34.71,32.77,10,10,Хмельницька
9,1982,10,0.057,273.10,27.66,36.79,32.23,10,10,Хмельницька


Реалізувати процедуру зміни індексів: в завантажених з NOAA даних області індексуються за англійською абеткою (Province 1 - Cherkasy), потрібно замінити індекси так, щоб області індексувалася за українською абеткою (1 область - Вінницька). 

In [4]:
# Процедура зміни індексів: NOAA індексує області за англійською абеткою, я перепризначаю ua_id так щоб області йшли за українською абеткою

UA_ALPHABET_ORDER = {
    "Вінницька":        1,
    "Волинська":        2,
    "Дніпропетровська": 3,
    "Донецька":         4,
    "Житомирська":      5,
    "Закарпатська":     6,
    "Запорізька":       7,
    "Івано-Франківська":8,
    "Київська":         9,
    "Київ (місто)":     10,
    "Кіровоградська":   11,
    "Кримська":         12,
    "Луганська":        13,
    "Львівська":        14,
    "Миколаївська":     15,
    "Одеська":          16,
    "Полтавська":       17,
    "Рівненська":       18,
    "Севастополь":      19,
    "Сумська":          20,
    "Тернопільська":    21,
    "Харківська":       22,
    "Херсонська":       23,
    "Хмельницька":      24,
    "Черкаська":        25,
    "Чернігівська":     26,
    "Чернівецька":      27,
}

def reindex_by_ua_alphabet(df):
    # Замінюю ua_id згідно з порядком української абетки
    df = df.copy()
    df["ua_id"] = df["region"].map(UA_ALPHABET_ORDER)
    print("Індекси змінено за українською абеткою:")
    print(df[["region", "ua_id"]].drop_duplicates().sort_values("ua_id").to_string(index=False))
    return df

df = reindex_by_ua_alphabet(df)
df.head()

Індекси змінено за українською абеткою:
           region  ua_id
        Вінницька      1
        Волинська      2
 Дніпропетровська      3
         Донецька      4
      Житомирська      5
     Закарпатська      6
       Запорізька      7
Івано-Франківська      8
         Київська      9
     Київ (місто)     10
   Кіровоградська     11
         Кримська     12
        Луганська     13
        Львівська     14
     Миколаївська     15
          Одеська     16
       Полтавська     17
       Рівненська     18
      Севастополь     19
          Сумська     20
    Тернопільська     21
       Харківська     22
       Херсонська     23
      Хмельницька     24
        Черкаська     25
     Чернігівська     26
      Чернівецька     27


,YEAR,WEEK,SMN,SMT,VCI,TCI,VHI,noaa_id,ua_id,region
0,1982,1,0.059,258.24,51.11,48.78,49.95,10,24,Хмельницька
1,1982,2,0.063,261.53,55.89,38.20,47.04,10,24,Хмельницька
2,1982,3,0.063,263.45,57.30,32.69,44.99,10,24,Хмельницька
3,1982,4,0.061,265.10,53.96,28.62,41.29,10,24,Хмельницька
4,1982,5,0.058,266.42,46.87,28.57,37.72,10,24,Хмельницька


Ряд VHI для області за вказаний рік;

In [5]:
# Ряд VHI для конкретної області за вказаний рік
def vhi_by_region_year(df, ua_id, year):
    result = df[(df["ua_id"] == ua_id) & (df["YEAR"] == year)][["WEEK", "VHI", "region"]]
    print(f"VHI для ua_id={ua_id} ({df[df['ua_id']==ua_id]['region'].iloc[0]}), рік {year}:")
    return result.reset_index(drop=True)

vhi_by_region_year(df, ua_id=22, year=2010)

VHI для ua_id=22 (Харківська), рік 2010:


,WEEK,VHI,region
0,1,49.85,Харківська
1,2,48.81,Харківська
2,3,48.93,Харківська
3,4,49.00,Харківська
4,5,47.35,Харківська
5,6,43.87,Харківська
6,7,40.51,Харківська
7,8,38.29,Харківська
8,9,36.99,Харківська
9,10,37.28,Харківська


Ряд VHI за вказаний діапазон років для вказаних областей;

In [6]:
#Ряд VHI за діапазон років для вказаних областей
def vhi_by_regions_years(df, ua_ids, year_from, year_to):
    result = df[
        (df["ua_id"].isin(ua_ids)) &
        (df["YEAR"] >= year_from) &
        (df["YEAR"] <= year_to)
    ][["ua_id", "region", "YEAR", "WEEK", "VHI"]]
    print(f"VHI для ua_ids={ua_ids}, роки {year_from}–{year_to}:")
    return result.reset_index(drop=True)

vhi_by_regions_years(df, ua_ids=[1, 5, 22], year_from=2000, year_to=2005)

VHI для ua_ids=[1, 5, 22], роки 2000–2005:


,ua_id,region,YEAR,WEEK,VHI
0,1,Вінницька,2000,1,24.22
1,1,Вінницька,2000,2,27.70
2,1,Вінницька,2000,3,30.68
3,1,Вінницька,2000,4,32.55
4,1,Вінницька,2000,5,34.73
...,...,...,...,...,...
871,22,Харківська,2005,48,45.15
872,22,Харківська,2005,49,49.03
873,22,Харківська,2005,50,46.52
874,22,Харківська,2005,51,46.12


Пошук екстремумів (min та max) для вказаних областей та років, середнього, медіани;

In [7]:
#Екстремуми, середнє та медіана для вказаних областей та років
def vhi_stats(df, ua_ids, year_from, year_to):
    subset = df[
        (df["ua_id"].isin(ua_ids)) &
        (df["YEAR"] >= year_from) &
        (df["YEAR"] <= year_to)
    ]
    stats = subset.groupby(["ua_id", "region"])["VHI"].agg(
        мін="min", макс="max", середнє="mean", медіана="median"
    ).round(2)
    print(f"Статистика VHI для ua_ids={ua_ids}, роки {year_from}–{year_to}:")
    return stats

vhi_stats(df, ua_ids=[1, 5, 22], year_from=2000, year_to=2024)

Статистика VHI для ua_ids=[1, 5, 22], роки 2000–2024:


,,мін,макс,середнє,медіана
ua_id,region,,,,
1,Вінницька,11.25,81.44,47.99,47.30
5,Житомирська,26.51,75.01,49.77,49.02
22,Харківська,9.36,91.42,48.71,47.96
